# 04 — Comparación entre corpus
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Vocabulario distintivo por fuente, índice de Jaccard entre
vocabularios. Validez externa del merge.
## Parámetros
- `DATA_DIR`, `SEED`, `OUT_DIR` (papermill).


In [0]:
# %% [code]
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
SEED = int(os.environ.get("SEED", 42))
OUT_DIR = Path(os.environ.get("OUT_DIR", "reports"))
np.random.seed(SEED)
df = pd.read_parquet(DATA_DIR / "processed" / "corpus_v1.parquet")
print(f"corpus: {len(df)} filas, {df['source'].nunique()} fuentes")


In [0]:
# %% [code]
# Vocabulario por fuente (top 20).
def vocab(texts, top=20):
    toks = Counter()
    for t in texts:
        toks.update((t or "").lower().split())
    return toks.most_common(top)
for src, sub in df.groupby("source"):
    print(f"--- {src} (n={len(sub)}) ---")
    for w, c in vocab(sub["text_clean"].fillna("")):
        print(f"  {w:20s} {c}")
    print()


In [0]:
# %% [code]
# Jaccard entre vocabularios.
def v_set(texts, min_freq=2):
    toks = Counter()
    for t in texts:
        toks.update((t or "").lower().split())
    return {w for w, c in toks.items() if c >= min_freq}
sources = list(df["source"].unique())
vocabs = {s: v_set(df[df["source"] == s]["text_clean"].fillna("")) for s in sources}
mat = pd.DataFrame(index=sources, columns=sources, dtype=float)
for a in sources:
    for b in sources:
        u = len(vocabs[a] | vocabs[b])
        mat.loc[a, b] = len(vocabs[a] & vocabs[b]) / u if u else 0
print(mat)


In [0]:
# %% [code]
# Heatmap de Jaccard.
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=ax)
ax.set_title("Jaccard entre vocabularios de fuentes (min_freq=2)")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_04_jaccard_fuentes.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()


In [0]:
# %% [code]
# Palabras distintivas por fuente (log-odds).
def log_odds(src_texts, other_texts, top=15):
    from collections import Counter
    a = Counter()
    b = Counter()
    for t in src_texts:
        a.update((t or "").lower().split())
    for t in other_texts:
        b.update((t or "").lower().split())
    all_words = set(a) | set(b)
    N_a, N_b = sum(a.values()), sum(b.values())
    out = []
    for w in all_words:
        if a[w] + b[w] < 5:
            continue
        # Suavizado Laplace.
        p_a = (a[w] + 1) / (N_a + len(all_words))
        p_b = (b[w] + 1) / (N_b + len(all_words))
        out.append((w, np.log(p_a / p_b)))
    out.sort(key=lambda x: -x[1])
    return out[:top], out[-top:]
for src in sources:
    sub = df[df["source"] == src]["text_clean"].fillna("").tolist()
    other = df[df["source"] != src]["text_clean"].fillna("").tolist()
    up, down = log_odds(sub, other, top=10)
    print(f"--- {src} ---")
    print("  más en esta fuente:", [w for w, _ in up])
    print("  menos en esta fuente:", [w for w, _ in down])
    print()


## Conclusiones
- Si el Jaccard entre fuentes es muy bajo (<0.10), el merge agrega poco
  valor. Hay que evaluar si conviene entrenar por fuente y luego promediar.
- Si el Jaccard es alto (>0.50), las fuentes son redundantes.
- Rango saludable esperado: 0.20-0.40.
